In [ ]:
import json
import sys
import os
from pathlib import Path

import torch
import numpy as np

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import transformers
transformers.logging.disable_progress_bar()

REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {props.name}  total={props.total_memory/1e9:.1f}GB  free={free/1e9:.1f}GB")
else:
    print("  WARNING: No CUDA — switch to the Python 3 (ipykernel) kernel and rerun.")


In [ ]:
DATASET_PATH = REPO_ROOT / "outputs" / "preference_dataset.jsonl"

pairs = []
with open(DATASET_PATH) as f:
    for line in f:
        if line.strip():
            pairs.append(json.loads(line))

print(f"Total preference pairs : {len(pairs)}")
print(f"Keys in each record    : {list(pairs[0].keys())}")

In [ ]:
prompt_lens   = [len(p["prompt"])   for p in pairs]
chosen_lens   = [len(p["chosen"])   for p in pairs]
rejected_lens = [len(p["rejected"]) for p in pairs]

for label, lens in [("prompt", prompt_lens), ("chosen", chosen_lens), ("rejected", rejected_lens)]:
    print(f"{label:>10}  min={min(lens):4d}  max={max(lens):4d}  mean={np.mean(lens):.0f} chars")

print("=" * 70)
print("\nSample pair (index 0):")
print(f"  prompt   : {pairs[0]['prompt'][:120]}")
print(f"  chosen   : {pairs[0]['chosen'][:120]}")
print(f"  rejected : {pairs[0]['rejected'][:120]}")

In [ ]:
TRAIN_PATH = REPO_ROOT / "outputs" / "preference_train.jsonl"
EVAL_PATH  = REPO_ROOT / "outputs" / "preference_eval.jsonl"
TEST_PATH  = REPO_ROOT / "outputs" / "preference_test.jsonl"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_pairs = load_jsonl(TRAIN_PATH)
eval_pairs  = load_jsonl(EVAL_PATH)
test_pairs  = load_jsonl(TEST_PATH)

print(f"Train: {len(train_pairs)} pairs  ->  {TRAIN_PATH}")
print(f"Eval:  {len(eval_pairs)}  pairs  ->  {EVAL_PATH}")
print(f"Test:  {len(test_pairs)}  pairs  ->  {TEST_PATH}")
print(f"Total: {len(train_pairs)+len(eval_pairs)+len(test_pairs)} pairs")


In [ ]:
BASE_MODEL = "deepseek-ai/deepseek-coder-6.7b-base"
OUTPUT_DIR = REPO_ROOT / "outputs" / "dpo_models"

DPO_CONFIG = {
    "beta":          0.1,
    "learning_rate": 1e-4,
    "num_epochs":    5,
    "batch_size":    1,   
    "warmup_steps":  100,
    "max_length":    2048,
    "use_lora":      True,
    "lora_rank":     16,
    "lora_alpha":    32,
    "lora_dropout":  0.05,
}

print("=" * 50)
print("SPEC-ALIGN DPO CONFIGURATION")
print("=" * 50)
print(f"Base model : {BASE_MODEL}")
print(f"Output dir : {OUTPUT_DIR}")
for k, v in DPO_CONFIG.items():
    print(f"  {k:<20} {v}")
print("=" * 50)


In [ ]:
from dpo_trainer import DPOTrainer

trainer = DPOTrainer(
    model_name    = BASE_MODEL,
    output_dir    = str(OUTPUT_DIR),
    beta          = DPO_CONFIG["beta"],
    learning_rate = DPO_CONFIG["learning_rate"],
    num_epochs    = DPO_CONFIG["num_epochs"],
    batch_size    = DPO_CONFIG["batch_size"],
    warmup_steps  = DPO_CONFIG["warmup_steps"],
    max_length    = DPO_CONFIG["max_length"],
    use_lora      = DPO_CONFIG["use_lora"],
    lora_rank     = DPO_CONFIG["lora_rank"],
    lora_alpha    = DPO_CONFIG["lora_alpha"],
    lora_dropout  = DPO_CONFIG["lora_dropout"],
    wandb_project = "tla_bench",
    wandb_run_name= "spec-align-dpo",
)

total_params     = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

print(f"\nDevice          : {trainer.device}")
print(f"Policy model    : {total_params:,} total  |  {trainable_params:,} trainable (LoRA)")
print(f"Reference model : {sum(p.numel() for p in trainer.ref_model.parameters()):,} params (frozen)")

In [ ]:
train_history = trainer.train(
    train_dataset_path = str(TRAIN_PATH),
    eval_dataset_path  = str(EVAL_PATH),
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Epochs trained       : {DPO_CONFIG['num_epochs']}")
print(f"Final epoch loss     : {train_history['epoch_losses'][-1]:.4f}")
if "eval_results" in train_history:
    er = train_history["eval_results"]
    print(f"Eval loss            : {er['eval_loss']:.4f}")
    print(f"Preference accuracy  : {er['preference_accuracy']:.4f}")
print("=" * 60)


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

batch_losses     = train_history["batch_losses"]
chosen_rewards   = train_history["chosen_rewards"]
rejected_rewards = train_history["rejected_rewards"]
reward_margins   = [c - r for c, r in zip(chosen_rewards, rejected_rewards)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(batch_losses, linewidth=1)
axes[0].set_title("DPO Loss (per batch)")
axes[0].set_xlabel("Batch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(reward_margins, linewidth=1, color="green")
axes[1].axhline(0, color="red", linestyle="--", linewidth=0.8)
axes[1].set_title("Reward Margin: chosen - rejected (per batch)")
axes[1].set_xlabel("Batch")
axes[1].set_ylabel("Reward Margin")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Spec-Prover DPO Training", fontsize=13, fontweight="bold")
plt.tight_layout()

fig_path = OUTPUT_DIR / "training_curves.png"
plt.savefig(str(fig_path), dpi=150)
plt.close(fig)
print(f"Saved: {fig_path}")


In [ ]:
checkpoint_path = trainer.save_checkpoint()

metadata = {
    "base_model"          : BASE_MODEL,
    "method"              : "DPO (Spec-Prover)",
    "total_pairs"         : len(train_pairs) + len(eval_pairs) + len(test_pairs),
    "train_pairs"         : len(train_pairs),
    "eval_pairs"          : len(eval_pairs),
    "test_pairs"          : len(test_pairs),
    "config"              : DPO_CONFIG,
    "final_train_loss"    : train_history["epoch_losses"][-1],
    "eval_loss"           : train_history.get("eval_results", {}).get("eval_loss"),
    "eval_preference_acc" : train_history.get("eval_results", {}).get("preference_accuracy"),
    "checkpoint_path"     : checkpoint_path,
}

meta_path = OUTPUT_DIR / "spec_prover_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Checkpoint : {checkpoint_path}")
print(f"Metadata         : {meta_path}")